In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

FEATURE_PATH = Path('shm_engineered_features.csv')
RESULTS_PATH = Path('shm_model_validation_results.csv')
dataset = pd.read_csv(FEATURE_PATH)
results = pd.read_csv(RESULTS_PATH)
print('Dataset:', dataset.shape)
display(results)

Dataset: (64, 29)


,model,MAPE,competition_score,MAE,RMSE,R2
0,Gradient Boosting,0.210194,0.789806,0.039728,0.074434,0.918926
1,Random Forest,0.246156,0.753844,0.040651,0.075466,0.916660
2,Extra Trees,0.250535,0.749465,0.040901,0.075140,0.917380
3,Ridge,0.890797,0.109203,0.075141,0.095654,0.866109
4,ElasticNet,0.891360,0.108640,0.064094,0.081027,0.903927
5,Dummy median,0.936741,0.063259,0.181302,0.293660,-0.261931


In [2]:
def linear(model):
    return Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', model)])

def tree(model):
    return Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', model)])

models = {
    'Dummy median': linear(DummyRegressor(strategy='median')),
    'Ridge': linear(Ridge(alpha=10.0)),
    'ElasticNet': linear(ElasticNet(alpha=0.01, l1_ratio=0.2, max_iter=20000, random_state=42)),
    'Random Forest': tree(RandomForestRegressor(n_estimators=300, min_samples_leaf=2, max_features=0.7, random_state=42, n_jobs=-1)),
    'Extra Trees': tree(ExtraTreesRegressor(n_estimators=300, min_samples_leaf=2, max_features=0.8, random_state=42, n_jobs=-1)),
    'Gradient Boosting': tree(GradientBoostingRegressor(n_estimators=150, learning_rate=0.03, max_depth=2, loss='huber', random_state=42)),
}
best_name = results.sort_values('competition_score', ascending=False).iloc[0]['model']
if best_name not in models: raise KeyError(f'Unknown selected model: {best_name}')
print('Selected model:', best_name)

Selected model: Gradient Boosting


In [3]:
TARGET = 'damage'
ID_COLUMNS = ['filename']
feature_columns = [c for c in dataset.columns if c not in ID_COLUMNS + [TARGET]]
X = dataset[feature_columns].replace([np.inf, -np.inf], np.nan)
y = dataset[TARGET].astype(float)
final_model = models[best_name]
final_model.fit(X, y)
training_predictions = final_model.predict(X)
print('Training rows:', len(X))
print('Training prediction range:', training_predictions.min(), training_predictions.max())

Training rows: 64
Training prediction range: 0.03508857394543954 0.8201382001380874


In [4]:
joblib.dump(final_model, 'shm_final_model.joblib')
pd.DataFrame({'feature':feature_columns}).to_csv('shm_final_features.csv', index=False)
pd.DataFrame({'filename':dataset['filename'], 'damage':y, 'training_prediction':training_predictions}).to_csv('shm_training_predictions.csv', index=False)
metadata = {'model_name':best_name, 'feature_count':len(feature_columns), 'training_samples':len(dataset), 'target':'damage', 'validation_metric':'max(0, 1 - MAPE)', 'features_file':'shm_final_features.csv'}
Path('shm_final_model_metadata.json').write_text(json.dumps(metadata, indent=2))
print('Saved shm_final_model.joblib')
print('Saved shm_final_features.csv')
print('Saved shm_final_model_metadata.json')

Saved shm_final_model.joblib
Saved shm_final_features.csv
Saved shm_final_model_metadata.json


In [5]:
reloaded = joblib.load('shm_final_model.joblib')
saved_features = pd.read_csv('shm_final_features.csv')['feature'].tolist()
assert saved_features == feature_columns
assert len(reloaded.predict(X)) == len(dataset)
print('Model reload check passed.')
print('Feature-order check passed.')
print('Ready for the inference notebook.')

Model reload check passed.
Feature-order check passed.
Ready for the inference notebook.
